# ECG Classification - PyTorch LSTM Model (v3-pytorch)**What's New in This Version:**- ✅ **PyTorch Implementation**: Converted from TensorFlow/Keras to PyTorch- ✅ **Fixed Data Leakage**: Scaler now fits ONLY on training data- ✅ **Stronger Regularization**: Increased dropout, added weight decay- ✅ **Direct ONNX Export**: Native PyTorch to ONNX conversion- ✅ **Manual Training Loop**: More control and transparency**Why This Fixes Overfitting:**1. **Data Leakage Fixed**: Previous version fit scaler on ALL data before splitting2. **Increased Dropout**: 0.3 → 0.5 (50% of neurons dropped during training)3. **Weight Decay**: L2 regularization added (weight_decay=1e-4)4. **Reduced Patience**: Early stopping patience 15 → 10 epochs5. **Better Validation**: Proper train/val/test isolation

## STEP 1: Import Libraries

In [ ]:
# Core librariesimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltfrom sklearn.preprocessing import StandardScalerfrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import (    confusion_matrix, ConfusionMatrixDisplay, classification_report,    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score)import warningswarnings.filterwarnings('ignore')# PyTorchimport torchimport torch.nn as nnimport torch.optim as optimfrom torch.utils.data import Dataset, DataLoader# Check PyTorch availabilityprint(f'PyTorch version: {torch.__version__}')print(f'CUDA available: {torch.cuda.is_available()}')device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f'Using device: {device}')# Set random seeds for reproducibilityRANDOM_STATE = 42torch.manual_seed(RANDOM_STATE)np.random.seed(RANDOM_STATE)if torch.cuda.is_available():    torch.cuda.manual_seed(RANDOM_STATE)

## STEP 2: Load ECG Dataset

In [ ]:
# Load the ECG dataset# Adjust path based on your environmentdf = pd.read_csv('../../dataset_aritmia_NEW.csv')print(f'Dataset Shape: {df.shape}')print(f'Columns: {df.columns.tolist()[:10]}... (showing first 10)')print(f'\nFirst few rows:')print(df.head())

## STEP 3: Exploratory Data Analysis

In [ ]:
# Check label distributionprint('Label Distribution:')print(df['label'].value_counts())print('\nPercentage:')print(df['label'].value_counts(normalize=True) * 100)# Visualize distributionfig, axes = plt.subplots(1, 2, figsize=(12, 4))df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['green', 'red'])axes[0].set_title('Distribution of ECG Labels')axes[0].set_xlabel('Label (0=Normal, 1=Abnormal)')axes[0].set_ylabel('Count')axes[0].set_xticklabels(['Normal (0)', 'Abnormal (1)'], rotation=0)df['label'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['green', 'red'])axes[1].set_title('Label Proportion')axes[1].set_ylabel('')plt.tight_layout()plt.show()

## STEP 4: Data Preprocessing - **FIXING DATA LEAKAGE****CRITICAL CHANGE:**Previous version had data leakage because it fit the scaler on ALL data before splitting.This version fixes it by:1. Split data FIRST2. Fit scaler ONLY on training data3. Transform validation and test using training statistics

In [ ]:
# Separate features and labelsX = df.drop('label', axis=1).valuesy = df['label'].valuesprint(f'Features shape: {X.shape}')print(f'Labels shape: {y.shape}')# =============================================================================# STEP 4.1: SPLIT FIRST (before any preprocessing)# =============================================================================# This is the CORRECT order to prevent data leakage# First split: 80% train, 20% temp (for val + test)X_train, X_temp, y_train, y_temp = train_test_split(    X, y,    test_size=0.2,    random_state=RANDOM_STATE,    stratify=y)# Second split: 50% of temp = 10% validation, 10% testX_val, X_test, y_val, y_test = train_test_split(    X_temp, y_temp,    test_size=0.5,    random_state=RANDOM_STATE,    stratify=y_temp)print(f'\nSplit sizes:')print(f'Training:   {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)')print(f'Validation: {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.1f}%)')print(f'Test:       {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)')# =============================================================================# STEP 4.2: NORMALIZE - FIT ONLY ON TRAINING DATA# =============================================================================# This prevents information leakage from validation/test setsscaler = StandardScaler()X_train_normalized = scaler.fit_transform(X_train)  # Fit on training onlyX_val_normalized = scaler.transform(X_val)          # Transform using training statsX_test_normalized = scaler.transform(X_test)        # Transform using training statsprint(f'\nNormalization (training data only):')print(f'Mean: {scaler.mean_[:5]}...')print(f'Std: {scaler.scale_[:5]}...')# Reshape for LSTM (samples, timesteps, features)X_train_reshaped = X_train_normalized.reshape(-1, 188, 1)X_val_reshaped = X_val_normalized.reshape(-1, 188, 1)X_test_reshaped = X_test_normalized.reshape(-1, 188, 1)print(f'\nReshaped for LSTM:')print(f'Training: {X_train_reshaped.shape}')print(f'Validation: {X_val_reshaped.shape}')print(f'Test: {X_test_reshaped.shape}')

## STEP 5: PyTorch Dataset and DataLoader

In [ ]:
class ECGDataset(Dataset):    """Custom PyTorch Dataset for ECG data"""    def __init__(self, X, y):        self.X = torch.FloatTensor(X)        self.y = torch.LongTensor(y)        def __len__(self):        return len(self.X)        def __getitem__(self, idx):        return self.X[idx], self.y[idx]# Create datasetstrain_dataset = ECGDataset(X_train_reshaped, y_train)val_dataset = ECGDataset(X_val_reshaped, y_val)test_dataset = ECGDataset(X_test_reshaped, y_test)# Create dataloadersBATCH_SIZE = 32train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)print(f'DataLoaders created:')print(f'Training batches: {len(train_loader)}')print(f'Validation batches: {len(val_loader)}')print(f'Test batches: {len(test_loader)}')

## STEP 6: PyTorch LSTM Model Architecture**Regularization Improvements:**- Dropout increased from 0.3 to 0.5 (more aggressive)- Bidirectional LSTM with proper dropout- Batch normalization after LSTM layers

In [ ]:
class ECG_LSTM(nn.Module):    """    Bidirectional LSTM for ECG Classification        Architecture:    - 2 Bidirectional LSTM layers (64 and 32 units per direction)    - Dropout 0.5 (increased from 0.3 to prevent overfitting)    - Batch Normalization    - Dense layers for classification    """    def __init__(self, input_size=1, hidden_size=64, num_layers=2, num_classes=2, dropout=0.5):        super(ECG_LSTM, self).__init__()                # First Bidirectional LSTM layer        self.lstm1 = nn.LSTM(            input_size=input_size,            hidden_size=hidden_size,            num_layers=1,            batch_first=True,            bidirectional=True,            dropout=0        )        self.bn1 = nn.BatchNorm1d(hidden_size * 2)        self.dropout1 = nn.Dropout(dropout)                # Second Bidirectional LSTM layer        self.lstm2 = nn.LSTM(            input_size=hidden_size * 2,            hidden_size=hidden_size // 2,            num_layers=1,            batch_first=True,            bidirectional=True,            dropout=0        )        self.bn2 = nn.BatchNorm1d(hidden_size)        self.dropout2 = nn.Dropout(dropout)                # Fully connected layers        self.fc1 = nn.Linear(hidden_size, 64)        self.bn3 = nn.BatchNorm1d(64)        self.dropout3 = nn.Dropout(dropout)                self.fc2 = nn.Linear(64, 32)        self.dropout4 = nn.Dropout(dropout * 0.8)  # Slightly less dropout                self.fc3 = nn.Linear(32, num_classes)                self.relu = nn.ReLU()        def forward(self, x):        # x shape: (batch, seq_len, input_size)                # First LSTM layer        lstm_out, _ = self.lstm1(x)        # lstm_out shape: (batch, seq_len, hidden_size*2)                # BatchNorm requires (batch, features, seq_len)        lstm_out = lstm_out.permute(0, 2, 1)        lstm_out = self.bn1(lstm_out)        lstm_out = lstm_out.permute(0, 2, 1)        lstm_out = self.dropout1(lstm_out)                # Second LSTM layer        lstm_out, (hidden, _) = self.lstm2(lstm_out)        # Take the last hidden state from both directions        # hidden shape: (2, batch, hidden_size//2) for bidirectional        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)        # hidden shape: (batch, hidden_size)                hidden = self.bn2(hidden)        hidden = self.dropout2(hidden)                # Fully connected layers        out = self.fc1(hidden)        out = self.relu(out)        out = self.bn3(out)        out = self.dropout3(out)                out = self.fc2(out)        out = self.relu(out)        out = self.dropout4(out)                out = self.fc3(out)                return out# Create modelmodel = ECG_LSTM(    input_size=1,    hidden_size=64,    num_layers=2,    num_classes=2,    dropout=0.5).to(device)print('Model Architecture:')print(model)print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters())}')print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}')